In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments

In [ ]:
dataset = load_dataset("burman-ai/The-Book-of-Mormon")

In [ ]:
dataset

In [ ]:
# The dataset only comes with a train split, so let's add a test set
dataset = dataset["train"].train_test_split(test_size=0.1)

In [ ]:
dataset

In [ ]:
dataset_train = dataset["train"]
dataset_test = dataset["test"]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["scripture_text"], truncation=True, padding="max_length")

In [ ]:
dataset_train = dataset_train.map(tokenize_function, batched=True)
dataset_test = dataset_test.map(tokenize_function, batched=True)

In [ ]:
# The tokens are in 'input_ids' field. Also important is 'attention_mask'.
# Let's keep only those fields and discard the rest
dataset_train = dataset_train.remove_columns([col for col in dataset_train.column_names if col not in ["input_ids", "attention_mask"]])
dataset_test = dataset_test.remove_columns([col for col in dataset_test.column_names if col not in ["input_ids", "attention_mask"]])

In [ ]:
# we add a labels field that is identical to input_ids for causal language modeling
# HF automatically shifts the labels over one position when computing the loss
dataset_train = dataset_train.map(
    lambda examples: {"labels": examples["input_ids"]},
    batched=True
)
dataset_test = dataset_test.map(
    lambda examples: {"labels": examples["input_ids"]},
    batched=True
)

In [ ]:
dataset_train

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gpt2")

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    fp16=True,
    logging_steps=100,
)

In [ ]:
from transformers import TrainerCallback

class SampleGenerationCallback(TrainerCallback):
    def __init__(self, tokenizer, sample_prompt, max_new_tokens=50):
        self.tokenizer = tokenizer
        self.sample_prompt = sample_prompt
        self.max_new_tokens = max_new_tokens

    def generate_sample(self, model):
        model.eval()
        inputs = self.tokenizer(self.sample_prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=self.max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
        text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        print("\nSample output:\n", text)
        model.train()

    def on_train_begin(self, args, state, control, **kwargs):
        print(f"\n=== Starting training ===")
        self.generate_sample(kwargs["model"])

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs:
            print(f"\nStep {state.global_step} | Loss: {logs['loss']:.4f}")
            self.generate_sample(kwargs["model"])


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_test,
    tokenizer=tokenizer,
    callbacks=[SampleGenerationCallback(tokenizer, sample_prompt="And it came to")]
)

In [ ]:
trainer.train()

In [ ]:
trainer.push_to_hub("finetuned-gpt2-book-of-mormon")